In [6]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import warnings
from IPython.display import display

# **Check DWH connection**

In [2]:
load_dotenv(find_dotenv())

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR") # <-- Читаем новую директорию

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

print("Testing connection to AWS RDS...")

try:
    with db_manager.get_dwh_connection() as conn:
        with conn.cursor() as cursor:
            
            cursor.execute("SELECT VERSION();")
            version = cursor.fetchone()
            
            print(f"Success! Connected to AWS RDS.")
            print(f"Database version: {version[0]}")
            
            cursor.execute("SHOW DATABASES;")
            databases = [db[0] for db in cursor.fetchall()]
            print(f"Available databases: {databases}")
            
except Exception as e:
    print(f"Test error: {e}")

Testing connection to AWS RDS...
Success! Connected to AWS RDS.
Database version: 8.4.8
Available databases: ['eliqsir_dwh', 'information_schema', 'mysql', 'performance_schema', 'sys']


# **DWH schema creation**

In [3]:
# 1. Load environment and set up paths
load_dotenv(find_dotenv())

# Fetch the sub-project directory where 'src' is located
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR")

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

# Add the sub-project directory to sys.path for imports
if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

# 2. Define the path to the schema.sql file using the new project directory
schema_file_path = os.path.join(IPAI_PROJECT_DIR, "src", "database", "schema.sql")

print(f"Looking for schema file at: {schema_file_path}")

try:
    # 3. Read the SQL file content
    with open(schema_file_path, 'r', encoding='utf-8') as file:
        sql_script = file.read()
    
    print("File loaded. Executing SQL statements on AWS RDS...")
    
    # 4. Split the script into individual statements
    # This bypasses the connector's 'multi' limitation
    sql_statements = sql_script.split(';')
    
    with db_manager.get_dwh_connection() as conn:
        # Use context manager for the cursor to ensure it closes properly
        with conn.cursor() as cursor:
            
            for statement in sql_statements:
                clean_statement = statement.strip()
                # Only execute if the statement isn't just empty space or newlines
                if clean_statement:
                    cursor.execute(clean_statement)
            
            # Commit the changes to the database
            conn.commit()
            
            print("Schema executed successfully!")
            
            # 5. Verification: Check if the tables were actually created
            cursor.execute("SHOW TABLES;")
            tables = [table[0] for table in cursor.fetchall()]
            
            print(f"\nTables currently in the 'eliqsir_dwh' database:")
            for table in tables:
                print(f" - {table}")
            
except FileNotFoundError:
    print(f"Error: Could not find the file 'schema.sql'. Check the path.")
except Exception as e:
    print(f"Execution error: {e}")

Looking for schema file at: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\ipai_project\src\database\schema.sql
File loaded. Executing SQL statements on AWS RDS...
Schema executed successfully!

Tables currently in the 'eliqsir_dwh' database:
 - dim_article
 - dim_drug
 - dim_protein
 - dim_structure
 - fact_bioactivity


# **ETL**

## **Extraction**

**To run the extraction correctly you need to perform following steps to have chembl db locally:**
1. Go to the official European Bioinformatics Institute FTP site: ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/

2. Download the file named chembl_36_sqlite.tar.gz 
3. Unzip/extract that file locally to data/ChEMBL project folder

### Extraction Layer: API Integration & Raw Data Acquisition

The **Extraction (E)** phase of the ELIQSIR ETL pipeline. This layer is responsible for querying external biological, chemical, and literature databases to retrieve the foundational datasets, saving the unformatted results directly into raw csv files.

The extraction process relies on four dedicated API/Database clients:

* **UniProt Extraction (`UniProtExtractor`):** Connects to the UniProt REST API to stream reviewed (Swiss-Prot) human proteome entries, capturing critical ML features like protein sequences, lengths, and family classifications.
* **ChEMBL Bioactivity Extraction (`ChemblExtractor`):** Queries a local ChEMBL SQLite database to extract high-quality bioactivity records (e.g., IC50, Ki) and drug metadata (SMILES, molecular weight, molecule type) for specific protein targets. Uses internal batching and Parquet caching for high-performance querying.
* **PDBe Structure Extraction (`PdbeExtractor`):** Fetches curated, high-resolution 3D protein structures mapping to our collected UniProt accessions via the PDBe Graph API.
* **PubMed Abstract Extraction (`PubMedExtractor`):** Uses the NCBI Entrez E-utilities API to fetch article metadata, full abstracts, and complete author lists based on PMIDs. Operates in configured batches with enforced delays to comply with NCBI rate-limiting policies.

**Output:** A set of raw, unaltered csv files (`chembl_raw.csv`, `pdbe_raw.csv`, `pubmed_raw.csv`, `uniprot_raw.csv`) stored in the local `data/csv/` directory, serving as the immutable starting point for the Transformation layer.

In [ ]:
# 1. Load environment variables
load_dotenv(find_dotenv())

# Fetch variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR")
DATA_DIR = os.getenv("DATA_DIR")
NCBI_EMAIL = os.getenv("NCBI_EMAIL")

# Ensure all critical variables are present
if not all([PROJECT_ROOT, IPAI_PROJECT_DIR, DATA_DIR, NCBI_EMAIL]):
    raise ValueError("Missing critical environment variables in the .env file!")

# Add the sub-project directory to sys.path so Python can find the 'src' module
if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

# 2. Set up the output directory
csv_dir = Path(DATA_DIR) / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

# Import the extractors from the src package
from src.etl.extraction import UniProtExtractor, ChemblExtractor, PdbeExtractor, PubMedExtractor

print(f"Starting ELIQSIR Extraction Pipeline...")
print(f"Output directory: {csv_dir}")

# -------------------------------------------------------------------
# STAGE 1: UniProt (The Foundation)
# -------------------------------------------------------------------
print("\n--- STAGE 1: Extracting UniProt Data ---")
uniprot_ext = UniProtExtractor()
df_uniprot = uniprot_ext.extract()

# Save raw CSV
df_uniprot.to_csv(csv_dir / "uniprot_raw.csv", index=False)

# Get the list of UniProt IDs for the next steps
uniprot_ids = df_uniprot['uniprot_id'].dropna().unique().tolist()

# TEST MODE: Keep only the first 50 proteins for a fast test run. 
# Remove or comment out the next line when you are ready to extract everything!
# uniprot_ids = uniprot_ids[:50] 
print(f"Proceeding with {len(uniprot_ids)} proteins for downstream extraction...")

# -------------------------------------------------------------------
# STAGE 2: ChEMBL (The Bioactivity & Drugs)
# -------------------------------------------------------------------
print("\n--- STAGE 2: Extracting ChEMBL Data ---")
# The ChEMBL extractor expects to find the 'chembl_36' folder inside this directory
chembl_base_dir = Path(DATA_DIR) / "ChEMBL" 

try:
    chembl_ext = ChemblExtractor(chembl_dir=chembl_base_dir)
    df_chembl = chembl_ext.extract(uniprot_ids=uniprot_ids)
    df_chembl.to_csv(csv_dir / "chembl_raw.csv", index=False)
    
    # Get PubMed IDs for Stage 3
    pubmed_ids = df_chembl['pubmed_id'].dropna().unique().tolist()
except FileNotFoundError as e:
    print(f"\nChEMBL Extraction Skipped: {e}")
    print("Please ensure your ChEMBL SQLite database is downloaded and extracted.")
    pubmed_ids = [] # Fallback if ChEMBL isn't downloaded yet

# -------------------------------------------------------------------
# STAGE 3: PubMed (The Articles)
# -------------------------------------------------------------------
print("\n--- STAGE 3: Extracting PubMed Data ---")
if pubmed_ids:
    pubmed_ext = PubMedExtractor(email=NCBI_EMAIL)
    df_pubmed = pubmed_ext.extract(pubmed_ids=pubmed_ids)
    df_pubmed.to_csv(csv_dir / "pubmed_raw.csv", index=False)
else:
    print("No PubMed IDs found (ChEMBL likely skipped). Skipping PubMed.")

# -------------------------------------------------------------------
# STAGE 4: PDBe (The 3D Structures)
# -------------------------------------------------------------------
print("\n--- STAGE 4: Extracting PDBe Data ---")
pdbe_ext = PdbeExtractor()
df_pdbe = pdbe_ext.extract(uniprot_ids=uniprot_ids)
df_pdbe.to_csv(csv_dir / "pdbe_raw.csv", index=False)

print(f"\nPipeline complete! All files saved to: {csv_dir}")

Starting ELIQSIR Extraction Pipeline...
Output directory: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv

--- STAGE 1: Extracting UniProt Data ---
2026-03-29 21:47:55  INFO      src.extraction.uniprot_extractor  Fetching UniProt data – organism_id=9606, reviewed=True
2026-03-29 21:48:07  INFO      src.extraction.uniprot_extractor  UniProt extraction complete – 20431 proteins retrieved.
Proceeding with 20431 proteins for downstream extraction...

--- STAGE 2: Extracting ChEMBL Data ---
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor  ============================================================
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor  ChEMBL SQLite Extractor initialized
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor    Version  : 36
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor    Database : data\ChEMBL\chembl_36\chembl_36_sqlite\chembl_36.db
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor   

## **Transformation**

### Transformation Layer: Data Cleaning & Dimensional Modeling

**Transformation (T)** phase of the ELIQSIR ETL pipeline. This layer acts as the bridge between raw extracted datasets (csv files) and our target Star Schema Data Warehouse, ensuring data quality, consistency, and relational integrity 

The transformation process is divided into two sequential stages:

### 1. Data Cleaning (`DataCleaner`)
Raw pandas DataFrames from the Extraction layer are cleaned using non-destructive operations:
* **Type Casting:** Converting strings to appropriate numeric types (e.g., floats for `molecular_weight` and `standard_value`, nullable `Int64` for `pubmed_id` and `year`).
* **Missing Value Handling:** Dropping rows missing critical identifiers (e.g., target or drug IDs), while safely preserving non-critical missing data as `pd.NA` (which translates to SQL `NULL`).
* **Normalization & Deduplication:** Stripping whitespace, removing exact duplicates, and canonicalizing identifiers (e.g., uppercase UniProt and PDB IDs).

### 2. Dimensional Building (`DimensionalModelBuilder`)
Cleaned DataFrames are restructured into a **Star Schema** consisting of Dimension and Fact tables:
* **Surrogate Key Generation:** Mapping natural string keys (e.g., `uniprot_id`, `drug_chembl_id`) to optimized integer surrogate keys (`protein_key`, `drug_key`) for faster SQL indexing.
* **Entity Integration:** Merging distinct data sources (e.g., joining PubMed abstracts with ChEMBL article metadata).
* **Fact Table Assembly:** Resolving foreign keys to securely link the central `fact_bioactivity` table to the surrounding dimensions (`dim_protein`, `dim_drug`, `dim_article`), ensuring no orphaned records are passed to the database.

**Output:** A set of normalized, relational DataFrames that will be saved into csv files.

In [4]:
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# Import transformation classes
from src.etl.transformation.cleaner import DataCleaner
from src.etl.transformation.dimensional_builder import DimensionalModelBuilder

# Load environment variables
load_dotenv()
BASE_DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
INPUT_DIR = BASE_DATA_DIR / "csv"
OUTPUT_DIR = BASE_DATA_DIR / "csv_cleaned"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize a dictionary to track row counts for the final report
report = {}

# 1. Load Raw Data
print(f"Loading raw CSV files from: {INPUT_DIR}")
raw_uniprot = pd.read_csv(INPUT_DIR / "uniprot_raw.csv")
raw_chembl = pd.read_csv(INPUT_DIR / "chembl_raw.csv")
raw_pdbe = pd.read_csv(INPUT_DIR / "pdbe_raw.csv")
raw_pubmed = pd.read_csv(INPUT_DIR / "pubmed_raw.csv")

report["raw"] = {
    "UniProt": len(raw_uniprot),
    "ChEMBL": len(raw_chembl),
    "PDBe": len(raw_pdbe),
    "PubMed": len(raw_pubmed)
}

# 2. Clean Data
print("Cleaning data...")
cleaner = DataCleaner()
clean_uniprot = cleaner.clean_uniprot(raw_uniprot)
clean_chembl = cleaner.clean_chembl(raw_chembl)
clean_pdbe = cleaner.clean_pdbe(raw_pdbe)
clean_pubmed = cleaner.clean_pubmed(raw_pubmed)

report["clean"] = {
    "UniProt": len(clean_uniprot),
    "ChEMBL": len(clean_chembl),
    "PDBe": len(clean_pdbe),
    "PubMed": len(clean_pubmed)
}

# 3. Build Dimensional Model
print("Building Star Schema dimensions and facts...")
builder = DimensionalModelBuilder()

# Dimensions MUST be built before the Fact table to generate surrogate keys
dim_protein = builder.build_dim_protein(clean_uniprot)
dim_drug = builder.build_dim_drug(clean_chembl)
dim_article = builder.build_dim_article(clean_chembl, clean_pubmed)
dim_structure = builder.build_dim_structure(clean_pdbe)

# Fact table resolves foreign keys
fact_bioactivity = builder.build_fact_bioactivity(clean_chembl)

report["dim"] = {
    "dim_protein": len(dim_protein),
    "dim_drug": len(dim_drug),
    "dim_article": len(dim_article),
    "dim_structure": len(dim_structure),
    "fact_bioactivity": len(fact_bioactivity)
}

# 4. Save to target directory
print(f"Saving cleaned dimensional files to: {OUTPUT_DIR}")
dim_protein.to_csv(OUTPUT_DIR / "dim_protein.csv", index=False)
dim_drug.to_csv(OUTPUT_DIR / "dim_drug.csv", index=False)
dim_article.to_csv(OUTPUT_DIR / "dim_article.csv", index=False)
dim_structure.to_csv(OUTPUT_DIR / "dim_structure.csv", index=False)
fact_bioactivity.to_csv(OUTPUT_DIR / "fact_bioactivity.csv", index=False)

# 5. Generate and print the metrics report
print("\n" + "="*60)
print("ELIQSIR TRANSFORMATION PIPELINE REPORT")
print("="*60)

print("\n--- 1. Data Cleaning (Raw -> Clean) ---")
for dataset in ["UniProt", "ChEMBL", "PDBe", "PubMed"]:
    raw_count = report['raw'][dataset]
    clean_count = report['clean'][dataset]
    dropped = raw_count - clean_count
    retention = (clean_count / raw_count * 100) if raw_count > 0 else 0
    print(f"{dataset:<10} | Raw: {raw_count:<8} | Clean: {clean_count:<8} | Dropped: {dropped:<6} | Retention: {retention:.1f}%")

print("\n--- 2. Dimensional Model (Final Tables) ---")
for table, count in report['dim'].items():
    print(f"{table:<20} | Rows: {count}")

print("="*60)
print("Pipeline execution completed successfully. Data saved.")

Loading raw CSV files from: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv


C:\Users\samos\AppData\Local\Temp\ipykernel_5676\978827551.py:22: DtypeWarning: Columns (16,19,22) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_chembl = pd.read_csv(INPUT_DIR / "chembl_raw.csv")


Cleaning data...
2026-03-31 00:07:52  INFO      src.etl.transformation.cleaner  UniProt cleaning complete - 20431 proteins retained.
2026-03-31 00:09:35  INFO      src.etl.transformation.cleaner  ChEMBL cleaning complete - 6441095 bioactivity records retained.
2026-03-31 00:09:54  INFO      src.etl.transformation.cleaner  PDBe cleaning complete - 225204 structure entries retained.
2026-03-31 00:09:59  INFO      src.etl.transformation.cleaner  PubMed cleaning complete - 36351 abstracts retained.
Building Star Schema dimensions and facts...
2026-03-31 00:09:59  INFO      src.etl.transformation.dimensional_builder  Building DimProtein - 20431 input rows.
2026-03-31 00:10:00  INFO      src.etl.transformation.dimensional_builder  DimProtein - 20431 new proteins added.
2026-03-31 00:10:00  INFO      src.etl.transformation.dimensional_builder  Building DimDrug.
2026-03-31 00:10:11  INFO      src.etl.transformation.dimensional_builder  DimDrug - 1509932 new drugs added.
2026-03-31 00:10:11  IN

### Transformation Results: Pre-Load Data Inspection & Star Schema Validation

To ensure the integrity of our ETL process, we visually inspect the generated Dimensional and Fact tables. 
Note: string identifiers (like `uniprot_id` or `drug_chembl_id`) have been supplemented or replaced with optimized integer surrogate keys (`protein_key`, `drug_key`, etc.) to prepare the data for efficient relational querying and possible HDF5 graph construction.

In [5]:
load_dotenv()
BASE_DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CLEANED_CSV_DIR = BASE_DATA_DIR / "csv_cleaned"

# List of tables in our Star Schema
tables = [
    "dim_protein",
    "dim_drug",
    "dim_article",
    "dim_structure",
    "fact_bioactivity"
]

print("Star Schema Validation: Inspecting Transformed Data\n")

# Loop through each file and display the first 10 rows
for table in tables:
    file_path = CLEANED_CSV_DIR / f"{table}.csv"
    
    if file_path.exists():
        print(f"--- {table.upper()} ---")
        # Read the first few rows to save memory
        df_head = pd.read_csv(file_path, nrows=10)
        # Use IPython's display for nice HTML rendering in Jupyter
        display(df_head)
        print("\n" + "-"*80 + "\n")
    else:
        print(f"Warning: File {file_path.name} not found in {CLEANED_CSV_DIR}.")

Star Schema Validation: Inspecting Transformed Data

--- DIM_PROTEIN ---


,protein_key,uniprot_id,gene_names,protein_name,sequence,sequence_length,protein_families
0,1,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,515,Cytochrome P450 family
1,2,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,SMIM26 family
2,3,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,NaN
3,4,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,MRWQEMGYIFYPRKLR,16,NaN
4,5,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,CD300 family
5,6,A0A0U1RRE5,NBDY LINC01420,Negative regulator of P-body association (P-bo...,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,68,NaN
6,7,A0A1B0GTW7,CIROP LMLN2,Ciliated left-right organizer metallopeptidase...,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788,Peptidase M8 family
7,8,A0A2R8Y7D0,TINCR LINC00036 NCRNA00036 PLAC2,Ubiquitin domain-containing protein TINCR (Pla...,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,87,NaN
8,9,A0A8I5KQE6,RPSA2 RPSA RPSAP58,Small ribosomal subunit protein uS2B (37 kDa l...,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,295,Universal ribosomal protein uS2 family
9,10,A0AV02,SLC12A8 CCC9,Solute carrier family 12 member 8 (Cation-chlo...,MTQMSQVQELFHEAAQQDALAQPQPWWKTQLFMWEPVLFGTWDGVF...,714,SLC12A transporter family



--------------------------------------------------------------------------------

--- DIM_DRUG ---


,drug_chembl_id,drug_key,drug_name,molecule_type,molecular_weight,canonical_smiles,standard_inchi_key
0,CHEMBL2322194,1,NaN,Small molecule,445.55,NS(=O)(=O)OC[C@@H]1C[C@@H](N2CCc3c(N[C@H]4CCc5...,AQGFWBQRVLPCIX-QXGSTGNESA-N
1,CHEMBL2017005,2,NaN,Small molecule,462.49,NS(=O)(=O)OC[C@H]1O[C@@H](n2cnc3c(N[C@H]4CCc5c...,PGAMXUGSHMQFKD-BPAMBQHCSA-N
2,CHEMBL1231160,3,PEVONEDISTAT,Small molecule,443.53,NS(=O)(=O)OC[C@@H]1C[C@@H](n2ccc3c(N[C@H]4CCc5...,MPUQHZXIXSTTDU-QXGSTGNESA-N
3,CHEMBL4226903,4,NaN,Small molecule,559.32,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,SRNWOUGRCWSEMX-ZIXUEBECSA-N
4,CHEMBL5205107,5,NaN,NaN,246.27,O=C(O)C1CCN(c2ncnc3[nH]ccc23)CC1,RUXSFWFRCSJWQK-UHFFFAOYSA-N
5,CHEMBL5208626,6,NaN,NaN,246.27,O=C(O)C1CCCN(c2ncnc3[nH]ccc23)C1,WWNAKDRFVWZIMY-UHFFFAOYSA-N
6,CHEMBL35505,7,DIHYDRALAZINE,Small molecule,190.21,N/N=c1\[nH][nH]/c(=N\N)c2ccccc12,VQKLRVZQQYVIJW-UHFFFAOYSA-N
7,CHEMBL5081193,8,NaN,Unknown,340.33,O=C(Nc1ccn(-c2ccnc(F)c2)n1)C1(c2ccccc2F)CC1,UWMKZOSESQVNAA-UHFFFAOYSA-N
8,CHEMBL5208423,9,NaN,NaN,321.41,O=C(Nc1nc(-c2ccncc2)cs1)C1(c2ccccc2)CC1,MFLSWUJMLWKBEZ-UHFFFAOYSA-N
9,CHEMBL5203167,10,NaN,NaN,331.35,O=C(Cc1ccccc1F)Nc1nc(-c2ccnc(F)c2)cs1,NLVGPTZAAUCTJS-UHFFFAOYSA-N



--------------------------------------------------------------------------------

--- DIM_ARTICLE ---


...


--------------------------------------------------------------------------------

--- DIM_STRUCTURE ---


,structure_key,protein_key,pdb_id,chain_id,resolution,coverage,method,unp_start,unp_end
0,1,8,7MRJ,A,2.120,0.989,X-ray diffraction,2,87
1,2,8,7MRJ,B,2.120,0.989,X-ray diffraction,2,87
2,3,11,2DIS,A,NaN,0.162,Solution NMR,150,245
3,4,14,4YO2,A,3.073,0.268,X-ray diffraction,110,341
4,5,15,7PVN,A,2.710,1.000,X-ray diffraction,1,1052
5,6,15,7PVN,B,2.710,1.000,X-ray diffraction,1,1052
6,7,15,9QH5,B,3.090,1.000,Electron Microscopy,1,1052
7,8,15,9QIC,B,3.290,1.000,Electron Microscopy,1,1052
8,9,15,9QIV,B,3.440,1.000,Electron Microscopy,1,1052
9,10,15,9QIM,B,3.570,1.000,Electron Microscopy,1,1052



--------------------------------------------------------------------------------

--- FACT_BIOACTIVITY ---


,activity_id,protein_key,drug_key,article_key,standard_type,standard_value,standard_units,pchembl_value,confidence_score,assay_type,assay_description,assay_organism
0,12645440,15,1,1,IC50,1800.0,nM,5.75,9,B,Inhibition of UBA6 (unknown origin),Homo sapiens
1,12645445,15,2,1,IC50,920.0,nM,6.04,9,B,Inhibition of UBA6 (unknown origin) in presenc...,Homo sapiens
2,18483955,15,3,2,IC50,1000.0,nM,NaN,9,B,Inhibition of UBA6 (unknown origin) assessed a...,Homo sapiens
3,18574594,45,4,3,Kd,150.0,nM,6.82,9,B,Binding affinity to human MDO2 by ITC,Homo sapiens
4,24775879,15,3,4,IC50,1800.0,nM,5.75,9,B,Inhibition of His-tagged UBA6 (unknown origin)...,Homo sapiens
5,24778420,45,5,5,IC50,300000.0,nM,NaN,9,T,Inhibition of N-terminal His-TEV-V5 tagged hum...,Homo sapiens
6,24778421,45,6,5,IC50,300000.0,nM,NaN,9,T,Inhibition of N-terminal His-TEV-V5 tagged hum...,Homo sapiens
7,24778436,45,7,5,IC50,500000.0,nM,NaN,9,T,Inhibition of human Mdo2,Homo sapiens
8,24792144,40,8,6,IC50,500000.0,nM,NaN,9,B,Inhibition of ELOVL7 (unknown origin) using C1...,Homo sapiens
9,24792145,40,9,6,IC50,100000.0,nM,NaN,9,B,Inhibition of ELOVL7 (unknown origin) using C1...,Homo sapiens



--------------------------------------------------------------------------------



## **Load**

### Loading Layer: Data Warehouse Integration

The **Loading (L)** phase of the ELIQSIR ETL pipeline. This layer is responsible for taking the fully prepared dimensional and fact csv files and inserting them into the target AWS RDS MySQL data warehouse.

The loading process is managed by the `WarehouseLoader` and relies on a custom connection manager:

* **Direct Database Connection:** Utilizes a custom `ConnectionManager` leveraging `mysql.connector` to establish secure, pooled connections directly to the AWS RDS instance, avoiding heavy ORM overhead.
* **Memory-Efficient Chunking:** Reads massive csv files (e.g., over 6 million bioactivity records) in configurable chunks (e.g., 10,000 rows at a time). This strictly bounds RAM usage and prevents database transaction timeouts.
* **Null Value Coercion:** Safely casts pandas missing data types (like `pd.NA` and `np.nan`) to native Python `None` objects within each chunk before execution. This ensures the database driver correctly inserts true SQL `NULL` values, preserving data integrity and preventing literal "nan" strings or type mismatch errors.
* **High-Performance Upserts:** Dynamically generates parameterized `INSERT ... ON DUPLICATE KEY UPDATE` SQL statements. Executes these via `cursor.executemany()` for lightning-fast, idempotent bulk inserts that seamlessly handle both initial loads and incremental updates without duplicating records.
* **Referential Integrity Enforcement:** Strictly dictates the load sequence. All Dimension tables (`dim_protein`, `dim_drug`, `dim_article`, `dim_structure`) are fully populated before the central Fact table (`fact_bioactivity`) to guarantee that all foreign key constraints are satisfied.

**Output:** A fully populated Star Schema residing in the AWS RDS instance. This relational database now serves as the ground-truth foundation for downstream Search and Machine Learning tasks.

In [ ]:
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
load_dotenv(Path(project_root) / ".env")
from src.etl.loading.warehouse_loader import WarehouseLoader

BASE_DATA_DIR = Path(os.getenv("DATA_DIR", Path(project_root) / "data"))
CLEANED_CSV_DIR = BASE_DATA_DIR / "csv_cleaned"

print(f"Connecting to AWS RDS and starting bulk load from: {CLEANED_CSV_DIR}")
print("This might take a while depending on your internet connection and chunk size...\n")

# chunksize=10000 is a safe default for remote databas to avoid packet size limits.
# If it runs too slow can be increased to 25000 or 50000 (in case if you do not see network errors).
loader = WarehouseLoader(chunksize=10000) 

try:
    results = loader.load_all(CLEANED_CSV_DIR)

    # Report generation
    print("\n" + "="*60)
    print("ELIQSIR LOADING PIPELINE REPORT (AWS RDS)")
    print("="*60)
    
    for table, rows in results.items():
        print(f"{table:<20} | Successfully inserted/updated: {rows} rows")
    
    total_rows = sum(results.values())
    print("-" * 60)
    print(f"TOTAL ROWS LOADED:   | {total_rows}")
    print("="*60)
    print("Data Warehouse loading completed successfully!")
    
except Exception as e:
    print(f"\nAn error occurred during loading: {e}")

Connecting to AWS RDS and starting bulk load from: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv_cleaned
This might take a while depending on your internet connection and chunk size...

2026-03-31 12:51:10  INFO      src.etl.loading.warehouse_loader  Beginning bulk warehouse load from directory: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv_cleaned
2026-03-31 12:51:10  INFO      src.etl.loading.warehouse_loader  Starting load for 'dim_protein' from C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv_cleaned\dim_protein.csv


2026-03-31 12:51:15  INFO      src.etl.loading.warehouse_loader  Finished loading 'dim_protein'. Total rows: 20431
2026-03-31 12:51:15  INFO      src.etl.loading.warehouse_loader  Starting load for 'dim_drug' from C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv_cleaned\dim_drug.csv
2026-03-31 12:51:21  INFO      src.etl.loading.warehouse_loader  ... loaded 50000 rows into 'dim_drug'
2026-03-31 12:51:26  INFO      src.etl.loading.warehouse_loader  ... loaded 100000 rows into 'dim_drug'
2026-03-31 12:51:31  INFO      src.etl.loading.warehouse_loader  ... loaded 150000 rows into 'dim_drug'
2026-03-31 12:51:37  INFO      src.etl.loading.warehouse_loader  ... loaded 200000 rows into 'dim_drug'
2026-03-31 12:51:43  INFO      src.etl.loading.warehouse_loader  ... loaded 250000 rows into 'dim_drug'
2026-03-31 12:51:49  INFO      src.etl.loading.warehouse_loader  ... loaded 300000 rows into 'dim_drug'
2026-03-31 12:51:54  INFO      src.etl.loading.warehouse_loader  ... loaded 350000 r

### Loading Results: Live Data Warehouse Verification

To confirm the successful execution of Loading layer, we query the live AWS RDS MySQL database directly. 
By fetching the first few rows from each table in our Star Schema, we verify that the data was not only transferred successfully over the network but also correctly parsed, typed, and indexed by the database engine.

In [8]:
# Suppress pandas warning about using raw DBAPI connections instead of SQLAlchemy
warnings.filterwarnings('ignore', category=UserWarning)
from src.database.connection_manager import db_manager

print("AWS RDS Verification: Querying Live Data\n")

# List of tables in our Star Schema
tables = [
    "dim_protein",
    "dim_drug",
    "dim_article",
    "dim_structure",
    "fact_bioactivity"
]

try:
    with db_manager.get_dwh_connection() as conn:
        for table in tables:
            print(f"--- LIVE TABLE: {table.upper()} ---")
            
            # Fetch only the first 10 rows directly from the cloud database
            query = f"SELECT * FROM `{table}` LIMIT 10;"
            df_head = pd.read_sql(query, con=conn)
            
            display(df_head)
            
except Exception as e:
    print(f"Database connection or query failed: {e}")

AWS RDS Verification: Querying Live Data

--- LIVE TABLE: DIM_PROTEIN ---


,protein_key,uniprot_id,gene_names,protein_name,sequence,sequence_length,protein_families
0,1,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,515,Cytochrome P450 family
1,2,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,SMIM26 family
2,3,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,None
3,4,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,MRWQEMGYIFYPRKLR,16,None
4,5,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,CD300 family
5,6,A0A0U1RRE5,NBDY LINC01420,Negative regulator of P-body association (P-bo...,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,68,None
6,7,A0A1B0GTW7,CIROP LMLN2,Ciliated left-right organizer metallopeptidase...,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788,Peptidase M8 family
7,8,A0A2R8Y7D0,TINCR LINC00036 NCRNA00036 PLAC2,Ubiquitin domain-containing protein TINCR (Pla...,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,87,None
8,9,A0A8I5KQE6,RPSA2 RPSA RPSAP58,Small ribosomal subunit protein uS2B (37 kDa l...,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,295,Universal ribosomal protein uS2 family
9,10,A0AV02,SLC12A8 CCC9,Solute carrier family 12 member 8 (Cation-chlo...,MTQMSQVQELFHEAAQQDALAQPQPWWKTQLFMWEPVLFGTWDGVF...,714,SLC12A transporter family


--- LIVE TABLE: DIM_DRUG ---


,drug_key,drug_chembl_id,drug_name,molecule_type,molecular_weight,canonical_smiles,standard_inchi_key
0,1,CHEMBL2322194,None,Small molecule,445.55,NS(=O)(=O)OC[C@@H]1C[C@@H](N2CCc3c(N[C@H]4CCc5...,AQGFWBQRVLPCIX-QXGSTGNESA-N
1,2,CHEMBL2017005,None,Small molecule,462.49,NS(=O)(=O)OC[C@H]1O[C@@H](n2cnc3c(N[C@H]4CCc5c...,PGAMXUGSHMQFKD-BPAMBQHCSA-N
2,3,CHEMBL1231160,PEVONEDISTAT,Small molecule,443.53,NS(=O)(=O)OC[C@@H]1C[C@@H](n2ccc3c(N[C@H]4CCc5...,MPUQHZXIXSTTDU-QXGSTGNESA-N
3,4,CHEMBL4226903,None,Small molecule,559.32,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,SRNWOUGRCWSEMX-ZIXUEBECSA-N
4,5,CHEMBL5205107,None,None,246.27,O=C(O)C1CCN(c2ncnc3[nH]ccc23)CC1,RUXSFWFRCSJWQK-UHFFFAOYSA-N
5,6,CHEMBL5208626,None,None,246.27,O=C(O)C1CCCN(c2ncnc3[nH]ccc23)C1,WWNAKDRFVWZIMY-UHFFFAOYSA-N
6,7,CHEMBL35505,DIHYDRALAZINE,Small molecule,190.21,N/N=c1\[nH][nH]/c(=N\N)c2ccccc12,VQKLRVZQQYVIJW-UHFFFAOYSA-N
7,8,CHEMBL5081193,None,Unknown,340.33,O=C(Nc1ccn(-c2ccnc(F)c2)n1)C1(c2ccccc2F)CC1,UWMKZOSESQVNAA-UHFFFAOYSA-N
8,9,CHEMBL5208423,None,None,321.41,O=C(Nc1nc(-c2ccncc2)cs1)C1(c2ccccc2)CC1,MFLSWUJMLWKBEZ-UHFFFAOYSA-N
9,10,CHEMBL5203167,None,None,331.35,O=C(Cc1ccccc1F)Nc1nc(-c2ccnc(F)c2)cs1,NLVGPTZAAUCTJS-UHFFFAOYSA-N


--- LIVE TABLE: DIM_ARTICLE ---


,article_key,pubmed_id,article_title,journal,year,abstract,doi,authors
0,1,23360215,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013,The labeling of proteins with small ubiquitin ...,10.1021/jm301420b,"da Silva, Sara R; Paiva, Stacey-Lynn; Lukkaril..."
1,2,28505447,Interrogating the Roles of Post-Translational ...,J Med Chem,2018,Post-translational modifications (PTMs) allot ...,10.1021/acs.jmedchem.6b01817,"Buuh, Zakey Yusuf; Lyu, Zhigang; Wang, Rongshe..."
2,3,29501416,Adenosine analogs bearing phosphate isosteres ...,Bioorg Med Chem,2018,The human O-acetyl-ADP-ribose deacetylase MDO1...,10.1016/j.bmc.2018.02.006,"Zhang, Yuezhou; Jumppanen, Mikael; Maksimainen..."
3,4,35131538,NAE modulators: A potential therapy for gastri...,Eur J Med Chem,2022,Neural precursor cell expressed developmentall...,10.1016/j.ejmech.2022.114156,"Liang, Qi; Liu, Maoyu; Li, Jian; Tong, Rongshe..."
4,5,35597097,"Design, synthesis and evaluation of inhibitors...",Bioorg Med Chem,2022,"A series of amino acid based 7H-pyrrolo[2,3-d]...",10.1016/j.bmc.2022.116788,"Sherrill, Lavinia M; Joya, Elva E; Walker, Ann..."
5,6,34748351,Discovery and Optimization of Pyrazole Amides ...,J Med Chem,2021,Accumulation of very long chain fatty acids (V...,10.1021/acs.jmedchem.1c00944,"Come, Jon H; Senter, Timothy J; Clark, Michael..."
6,7,29928781,NVP-BHG712: Effects of Regioisomers on the Aff...,ChemMedChem,2018,Erythropoietin-producing hepatocellular (EPH) ...,10.1002/cmdc.201800398,"Tröster, Alix; Heinzlmeir, Stephanie; Berger, ..."
7,8,28408219,Developing DYRK inhibitors derived from the me...,Bioorg Med Chem Lett,2017,A structure-activity relationship has been dev...,10.1016/j.bmcl.2017.03.037,"Shaw, Simon J; Goff, Dane A; Lin, Nan; Singh, ..."
8,9,28256837,Determination of Gymnemic Acid I as a Protein ...,J Nat Prod,2017,The plant Gymnema sylvestre has been used wide...,10.1021/acs.jnatprod.6b00793,"Capolupo, Angela; Esposito, Roberta; Zampella,..."
9,10,30929949,Profiling withanolide A for therapeutic target...,Bioorg Med Chem,2019,To identify new potential therapeutic targets ...,10.1016/j.bmc.2019.03.022,"Crane, Erika A; Heydenreuter, Wolfgang; Beck, ..."


--- LIVE TABLE: DIM_STRUCTURE ---


,structure_key,protein_key,pdb_id,chain_id,resolution,coverage,method,unp_start,unp_end
0,1,8,7MRJ,A,2.120,0.989,X-ray diffraction,2,87
1,2,8,7MRJ,B,2.120,0.989,X-ray diffraction,2,87
2,3,11,2DIS,A,NaN,0.162,Solution NMR,150,245
3,4,14,4YO2,A,3.073,0.268,X-ray diffraction,110,341
4,5,15,7PVN,A,2.710,1.000,X-ray diffraction,1,1052
5,6,15,7PVN,B,2.710,1.000,X-ray diffraction,1,1052
6,7,15,9QH5,B,3.090,1.000,Electron Microscopy,1,1052
7,8,15,9QIC,B,3.290,1.000,Electron Microscopy,1,1052
8,9,15,9QIV,B,3.440,1.000,Electron Microscopy,1,1052
9,10,15,9QIM,B,3.570,1.000,Electron Microscopy,1,1052


--- LIVE TABLE: FACT_BIOACTIVITY ---


,activity_id,protein_key,drug_key,article_key,standard_type,standard_value,standard_units,pchembl_value,confidence_score,assay_type,assay_description,assay_organism
0,31863,2620,632636,17930,IC50,100000.0,nM,NaN,8,B,Inhibitory concentration against human DNA top...,None
1,31864,14201,1466864,36065,IC50,2500.0,nM,5.60,8,B,In vivo inhibitory activity against human Hepa...,None
2,31866,14201,1466865,36065,IC50,9000.0,nM,5.05,8,B,In vivo inhibitory activity against human Hepa...,None
3,31874,2143,368569,6824,IC50,6000.0,nM,5.22,8,A,Inhibition of cytochrome P450 1A2 of isolated ...,Cavia porcellus
4,31875,2345,368569,6824,IC50,37000.0,nM,4.43,8,A,Inhibition of cytochrome P450 3A4 of isolated ...,Cavia porcellus
5,31876,2637,368569,6824,IC50,24000.0,nM,4.62,8,A,Inhibition of cytochrome P450 2C9 of isolated ...,Cavia porcellus
6,31889,1854,77825,6819,Ki,6000.0,nM,5.22,8,B,Inhibitory activity against human carbonic anh...,None
7,31890,1855,77825,6819,Ki,70.0,nM,7.16,8,B,Inhibitory activity against human carbonic anh...,None
8,31891,6723,77825,6819,Ki,285.0,nM,6.54,8,B,Inhibitory activity against human carbonic anh...,None
9,31893,1854,77810,6819,Ki,6.0,nM,8.22,8,B,Inhibitory activity against human carbonic anh...,None
